1.1Self_attention with Weights

In [2]:
import torch

In [3]:
inputs=torch.tensor([[0.43, 0.15, 0.89], 
                     [0.55, 0.87, 0.66],
                     [0.57, 0.85, 0.64],
                     [0.22, 0.58, 0.33],
                     [0.77, 0.25, 0.10],
                     [0.05, 0.80, 0.55]]
                    )
print(inputs.shape)  # torch.Size([6, 3])

torch.Size([6, 3])


In [4]:
d_in=inputs.shape[1]
d_out=2
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out),requires_grad=False)
W_key=torch.nn.Parameter(torch.rand(d_in, d_out),requires_grad=False)
W_value=torch.nn.Parameter(torch.rand(d_in, d_out),requires_grad=False)

In [5]:
x_2=inputs[1]
query_2=x_2 @ W_query
key_2=x_2 @ W_key
value_2=x_2 @ W_value
print(query_2)

tensor([0.4306, 1.4551])


In [6]:
keys=inputs @ W_key
values=inputs @ W_value
print(keys.shape)
print(values.shape)

torch.Size([6, 2])
torch.Size([6, 2])


In [7]:
attention_scores=query_2 @ keys.T
print(attention_scores.shape)  # torch.Size([6])

torch.Size([6])


In [8]:
d_k=keys.shape[-1]
attention_weights=torch.nn.functional.softmax(attention_scores/d_k**0.5,dim=-1)
print(attention_weights)  # torch.Size([6])

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [9]:
context_vector=attention_weights @ values
print(context_vector)  # torch.Size([2])

tensor([0.3061, 0.8210])


1.2 Self Attention Class

In [10]:
import torch.nn as nn

In [11]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in,d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
    def forward(self,x):
        keys=x@self.W_key
        values=x@self.W_value
        query=x@self.W_query
        attention_scores=query @ keys.T
        attention_weights=torch.softmax(attention_scores/keys.shape[-1]**0.5,dim=-1)
        context_vector=attention_weights @ values
        return context_vector

In [12]:
torch.manual_seed(123)
sa_va=SelfAttention_v1(d_in,d_out)
print(sa_va(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [13]:
class SelfAttention_v2(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out,bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out,bias=qkv_bias)
    def forward(self,x):
        keys=self.W_key(x)
        values=self.W_value(x)
        query=self.W_query(x)
        attention_scores=query @ keys.T
        attention_weights=torch.softmax(attention_scores/keys.shape[-1]**0.5,dim=-1)
        context_vector=attention_weights @ values
        return context_vector

In [14]:
torch.manual_seed(123)
sa_va2=SelfAttention_v2(d_in,d_out)
print(sa_va2(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


1.3 Mask Self Attention

In [15]:
queries=sa_va2.W_query(inputs)
keys=sa_va2.W_key(inputs)
attention_scores=queries @ keys.T
attention_weights=torch.softmax(attention_scores/keys.shape[-1]**0.5,dim=-1)
print(attention_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


In [16]:
context_length=attention_scores.shape[0]
mask_simple=torch.tril(torch.ones(context_length,context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [17]:
mask_simple=mask_simple*attention_weights
print(mask_simple)

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


In [18]:
row_sums=mask_simple.sum(dim=1, keepdim=True)
mask_simple_normalized=mask_simple/row_sums
print(mask_simple_normalized)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)


1.4 Mask Self Attention Class

In [19]:
batch=torch.stack((inputs,inputs),dim=0)
print(batch.shape)  # torch.Size([2, 6, 3])

torch.Size([2, 6, 3])


In [20]:
class CausalSelfAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,qkv_bias=False):
        super().__init__()
        self.d_out=d_out
        self.W_query = nn.Linear(d_in, d_out,bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out,bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(#将它注册为模型的缓冲区：不作为参数训练，但会随模型迁移到 GPU，默认也会存进 state_dict
            'mask', torch.triu(torch.ones(context_length, context_length),diagonal=1)
        )
    def forward(self,x):
        b,num_tokens,d_in=x.shape
        keys=self.W_key(x)
        values=self.W_value(x)
        query=self.W_query(x)

        attention_scores=query @ keys.transpose(1,2)
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
        #.bool()：将 1 变成 True，0 变成 False
        #masked_fill_：把掩码为 True 的位置原地改成 -∞。末尾的 _ 表示直接修改原张量。
        attention_weights=torch.softmax(attention_scores/keys.shape[-1]**0.5,dim=-1)
        attention_weights=self.dropout(attention_weights)
        cnotext_vector=attention_weights @ values
        return cnotext_vector

In [21]:
torch.manual_seed(123)
context_length=batch.shape[1]
ca=CausalSelfAttention(d_in,d_out,context_length,dropout=0.0)
context_vector=ca(batch)
print(context_vector.shape)  # torch.Size([2, 6, 2])


torch.Size([2, 6, 2])


1.5 Multi-head Attention

In [22]:
class MultiHeadAttentionWraper(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        self.heads=nn.ModuleList([CausalSelfAttention(d_in,d_out,context_length,dropout,qkv_bias) for _ in range(num_heads)])
    def forward(self,x):
        head_outputs=[head(x) for head in self.heads]
        return torch.cat(head_outputs,dim=-1)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_head = d_out // num_heads
        self.d_out=d_out
        self.W_k=nn.Linear(d_in, d_out,bias=qkv_bias)
        self.W_q=nn.Linear(d_in, d_out,bias=qkv_bias)
        self.W_v=nn.Linear(d_in, d_out,bias=qkv_bias)
        self.out=nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask', torch.triu(torch.ones(context_length, context_length),diagonal=1)
        )

    def forward(self,x):
        b,num_tokens,d_in=x.shape
        
        keys=self.W_k(x).view(b,num_tokens,self.num_heads,self.d_head).transpose(1,2)
        queries=self.W_q(x).view(b,num_tokens,self.num_heads,self.d_head).transpose(1,2)
        values=self.W_v(x).view(b,num_tokens,self.num_heads,self.d_head).transpose(1,2)

        attention_scores=queries @ keys.transpose(2,3)
        maskbool=self.mask.bool()[:num_tokens,:num_tokens]
        attention_scores.masked_fill_(maskbool,-torch.inf)

        attention_weights=torch.softmax(attention_scores/keys.shape[-1]**0.5,dim=-1)
        attention_weights=self.dropout(attention_weights)
        context=attention_weights @ values
        context=context.transpose(1,2).contiguous().view(b,num_tokens,self.d_out)#contiguous()：让数据连续。
        return self.out(context)